# Inference and Sampling


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
from course_tools import DEFAULT_CHAT_MESSAGES, ensure_demo_checkpoint, format_messages, generate_text, prefill_prompt, decode_next_token

LECTURE_NOTE_TITLE = 'Inference and Sampling'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')
bundle = ensure_demo_checkpoint(steps=40)
model = bundle['model']
tokenizer = bundle['tokenizer']
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)


Lecture note: Inference and Sampling


## Demo A: prefill then decode


In [2]:
prompt_ids, past_kvs = prefill_prompt(model, tokenizer, prompt)
print('prefill tokens:', len(prompt_ids))
print('cache shapes:', [kv[0].shape for kv in past_kvs])
next_id, past_kvs, _ = decode_next_token(model, prompt_ids[-1], past_kvs, temperature=0.8, top_k=8)
print('first decoded token:', next_id, repr(tokenizer.decode([next_id])))


prefill tokens: 64
cache shapes: [torch.Size([1, 4, 64, 12]), torch.Size([1, 4, 64, 12])]
first decoded token: 35 'n'


## Demo B: greedy decoding


In [3]:
print(repr(generate_text(model, tokenizer, prompt, max_new_tokens=60, temperature=0.0, top_k=None)))


'nintintinininininininininininininininininininininininininini'


## Demo C: temperature and top-k change the traversal of the same logits


In [4]:
for temperature, top_k in [(0.4, 4), (0.8, 8), (1.2, 12)]:
    print({'temperature': temperature, 'top_k': top_k})
    print(repr(generate_text(model, tokenizer, prompt, max_new_tokens=60, temperature=temperature, top_k=top_k)))
    print('-' * 80)


{'temperature': 0.4, 'top_k': 4}
'intt tintininttintinintintnnsss tttins tiinnininnsinns tntin'
--------------------------------------------------------------------------------
{'temperature': 0.8, 'top_k': 8}
'\ntntanitntentsrinanaarntntisnnedtinstnsninttin te tnttintine'
--------------------------------------------------------------------------------
{'temperature': 1.2, 'top_k': 12}
'srtdrniisiitimmtam stssdntedntoaccdtsnce ehentis tsdnrentini'
--------------------------------------------------------------------------------


## Demo D: top-p is a common extension, even though this runtime uses top-k


This repo implements temperature and top-k.
Top-p or nucleus sampling is the standard next extension to discuss in lecture.


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_generate.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/engine.py
